# Shapash Report

The Shapash Report feature allows data scientists to deliver to anyone interested in their project **a document that freezes different aspects of their work as a basis for an audit report**.

The shapash `generate_report` method generates an HTML report for your project.  
The report is generated as a single HTML file with embedded branding (including the logo), so the file can be moved and opened locally without breaking the logo.  
Some interactive resources may still rely on CDN assets depending on your environment, so an internet connection can still be required for full interactivity.  

The report contains the following information:
1. General information about the project
2. Description of the dataset used
3. Documentation about data preparation and feature engineering
4. Details about your model (library, parameters...)
5. Exploration of the data with a focus on the difference between train and test sets
6. Global explainability of the model
7. Model performance

The first three points are generated using a YAML file that the user should fill. An example is available [here](https://github.com/MAIF/shapash/blob/master/tutorial/report/config/project_information.yml).

This tutorial presents an example of how to generate the Shapash Report.

Content:
- Set up an example project
- Create and fill your project information that will be displayed in the report
- Generate the base Shapash Report
- Go further: generate a custom report with a custom YAML configuration

Data from Kaggle [House Prices](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)

> Note: Open the generated HTML file in a browser to view the final report.

In [ ]:
import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go
from category_encoders import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

## Building Supervized Model 

In [ ]:
from shapash.data.data_loader import data_loading
house_df, house_dict = data_loading('house_prices')
y_df=house_df['SalePrice']
X_df=house_df[house_df.columns.difference(['SalePrice'])]

In [ ]:
from category_encoders import OrdinalEncoder

categorical_features = [col for col in X_df.columns if X_df[col].dtype == 'object']

encoder = OrdinalEncoder(
    cols=categorical_features,
    handle_unknown='ignore',
    return_df=True).fit(X_df)

X_df = encoder.transform(X_df)

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X_df, y_df, train_size=0.75, random_state=1)

In [ ]:
regressor = RandomForestRegressor(n_estimators=50).fit(Xtrain, ytrain)

In [ ]:
y_pred = pd.DataFrame(regressor.predict(Xtest),columns=['pred'], index=Xtest.index)

## Fill your project information

**The next step is to create a YML file containing information about your project.**  

We will use the example file available [here](https://github.com/MAIF/shapash/blob/master/tutorial/report/utils/project_info.yml).  
**You are welcome to use this file as a template for your own report.**  

We display the information contained in the YML file below :

In [ ]:
import yaml

with open(r'config/project_information.yml') as file:
    project_info = yaml.full_load(file)

print(yaml.dump(project_info, sort_keys=False))

---
**If you want to create your own custom file :**

The keys of the YML file are the titles of the different sections in the report.  
The YML file must then respect the following format:

```yaml
Title of section 1:  
    property1 name: property1 value 
    property2 name: property2 value 
    ...
Title of section 2:  
    property1 name: property1 value 
    ...
```
> Note that the **date** can be computed automatically using the *auto* property value (see example above)

## Generate your report

### Declare and compile SmartExplainer object

In [ ]:
from shapash import SmartExplainer
from shapash.report.blocks import ReportBlockMixin, block

In [ ]:
xpl = SmartExplainer(
    model=regressor,
    preprocessing=encoder, # Optional: compile step can use inverse_transform method
    features_dict=house_dict  # optional parameter, specifies label for features name 
) 

In [ ]:
xpl.compile(x=Xtest, y_pred=y_pred, y_target=ytest)

At this step the model can be checked and inspected using different methods of the SmartExplainer object we just created.  

Please refer to the other tutorials for more information.

### Generate the base Shapash Report

Next we can generate the report using the `generate_report` method of our SmartExplainer object.

We need to pass `x_train`, `y_train` and `y_test` parameters in order to explore the data used when training the model.

Please refer to the documentation for a full description of the parameters.


In [ ]:
xpl.generate_report(
    output_file='output/report.html', 
    project_info_file='config/project_information.yml',
    x_train=Xtrain,
    y_train=ytrain,
    y_test=ytest,
    title_story="House prices report",
    title_description="""This document is a data science report of the kaggle house prices tutorial project. 
        It was generated using the Shapash library.""",
    metrics=[
        {
            'path': 'sklearn.metrics.mean_absolute_error',
            'name': 'Mean absolute error', 
        },
        {
            'path': 'sklearn.metrics.mean_squared_error',
            'name': 'Mean squared error',
        }
    ]
)

> Note: `generate_report` no longer includes the deprecated `kernel_name` parameter.

## Customize your own report

Now let's customize our report by adding user-defined blocks and a custom YAML layout.

To do so:
- Start from the default report YAML configuration.
- Add custom block types in YAML (for example, `user_note` or `prediction_diagnostics`).
- Implement matching methods named `block_<type>` in a class inheriting from `ReportBlockMixin`.
- Pass your custom file with `yaml_path="path/to/your/custom/report.yml"`.
- Pass your block class instance with `block_instance=YourCustomBlocks()`.

For this tutorial, we use `config/default_report_custom.yml` and add a custom diagnostics section rendered by a user block.

Next, we define the custom block class and generate the custom report:

In [ ]:
class UserReportBlocks(ReportBlockMixin):
    @block
    def block_user_note(
        self,
        title: str = "Analyst note",
        body: str = "This report includes a custom user cell.",
    ):
        return title, [pn.pane.Markdown(body)]

    @block
    def block_prediction_diagnostics(
        self,
        title: str = "Prediction diagnostics",
        color_feature: str | None = None,
    ):
        if self.y_test is None or self.y_pred is None:
            return title, [pn.pane.Markdown("Prediction diagnostics requires both y_test and y_pred.")]

        diagnostics = pd.DataFrame(
            {
                "actual": pd.Series(self.y_test).reset_index(drop=True),
                "predicted": pd.Series(self.y_pred).reset_index(drop=True),
            }
        )
        diagnostics["residual"] = diagnostics["actual"] - diagnostics["predicted"]
        diagnostics["abs_error"] = diagnostics["residual"].abs()

        if color_feature and self.x_init is not None and color_feature in self.x_init.columns:
            diagnostics[color_feature] = pd.Series(self.x_init[color_feature]).reset_index(drop=True)
            scatter = px.scatter(
                diagnostics,
                x="actual",
                y="predicted",
                color=color_feature,
                hover_data=["residual", "abs_error"],
                title="Actual vs predicted",
                labels={"actual": "Actual", "predicted": "Predicted"},
            )
        else:
            scatter = px.scatter(
                diagnostics,
                x="actual",
                y="predicted",
                color="abs_error",
                color_continuous_scale="Tealgrn",
                hover_data=["residual", "abs_error"],
                title="Actual vs predicted",
                labels={
                    "actual": "Actual",
                    "predicted": "Predicted",
                    "abs_error": "Absolute error",
                },
            )

        min_axis = min(diagnostics["actual"].min(), diagnostics["predicted"].min())
        max_axis = max(diagnostics["actual"].max(), diagnostics["predicted"].max())
        scatter.add_trace(
            go.Scatter(
                x=[min_axis, max_axis],
                y=[min_axis, max_axis],
                mode="lines",
                line={"dash": "dash", "color": "#666666"},
                name="Ideal fit",
                showlegend=False,
            )
        )
        scatter.update_layout(margin=dict(l=20, r=20, t=50, b=20))

        residual_hist = px.histogram(
            diagnostics,
            x="residual",
            nbins=30,
            title="Residual distribution",
            labels={"residual": "Actual - Predicted"},
            color_discrete_sequence=["#2E8B57"],
        )
        residual_hist.add_vline(x=0, line_dash="dash", line_color="#666666")
        residual_hist.update_layout(margin=dict(l=20, r=20, t=50, b=20))

        summary = pn.pane.Markdown(
            (
                "**Quick diagnostics:** "
                f"MAE = {diagnostics['abs_error'].mean():.2f}, "
                f"mean residual = {diagnostics['residual'].mean():.2f}, "
                f"max absolute error = {diagnostics['abs_error'].max():.2f}"
            )
        )

        scatter_pane = pn.pane.Plotly(
            scatter,
            config={"displayModeBar": False, "responsive": True},
            sizing_mode="stretch_width",
            height=360,
        )
        residual_pane = pn.pane.Plotly(
            residual_hist,
            config={"displayModeBar": False, "responsive": True},
            sizing_mode="stretch_width",
            height=360,
        )
        charts = pn.Row(scatter_pane, residual_pane, sizing_mode="stretch_width")
        return title, [summary, charts]

xpl.generate_report(
    output_file='output/custom_report.html',
    project_info_file='config/project_information.yml',
    x_train=Xtrain,
    y_train=ytrain,
    y_test=ytest,
    yaml_path='config/default_report_custom.yml',
    block_instance=UserReportBlocks(),
)

The custom report uses block types declared in `config/default_report_custom.yml`.
For each custom `type`, Shapash resolves a method named `block_<type>` from the provided `block_instance`.

After running the cell above, open `output/custom_report.html` in your browser.